In [23]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 5, 7, 11])

nb_name = "01_assembling_anndata.ipynb"
_compare_version(nb_name)

# 01 - Assembling or loading anndata object
<hr style="border:2px solid black"> </hr>

## 1 - Description
This notebook is dedicated to load or create an anndata object. The anndata object is prepared for the following analysis notebooks and finally stored as an `.h5ad` file. Based on the available data files there are multiple options to create the anndata object.

### Available options:
#### 1. `.h5ad` file
Choose this option if you have one or more `.h5ad` file(s). The file could be provided by a preprocessing pipeline, a public dataset or a preceeding analysis.

#### 2. star solo quant folder
This option is intended to assemble anndata object from the standard [star solo](https://github.com/alexdobin/STAR/tree/master) output folder (`quant/`). This is done by scaning through the folder structure and using the `*_matrix.mtx`, `*_barcodes.tsv` and `*_genes.tsv` to create an anndata object per sample. The sample anndata objects are finally combined.

#### 3. .mtx, barcode.tsv, genes.tsv
Choose this option if you have the expression matrix in `.mtx` format, a file containing the barcodes (`*_barcodes.tsv`) and a file containing the genes (`*_genes.tsv`). Use this option for cases with the aforementioned three files available e.g. from a public dataset.

#### 4. convert from R object
This option should be used if the data was processed using 'R'. This can either be a `.rds` or `.robj` file.

______

## 2 - Setup

In [3]:
import pandas as pd
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.utils.assemblers as assembler

sctoolbox.settings.settings_from_config("config.yaml", key="01")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


[WARNING] Log file '../logs/01_log.txt' already exists. The file will be overwritten since 'overwrite_log' is set to True.


,Name,Version
0,Python,"3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]"
1,sctoolbox,NA
2,numpy,2.3.5
3,anndata,0.12.6
4,pandas,2.3.3
5,scanpy,1.11.5


___

## 3 - Read in data
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [4]:
# For option 1: The path to an existing .h5ad file.
# A list or dict for multiple files e.g. {"rep_1": "1.h5ad", "rep_2": "2.h5ad"}.
path_h5ad = {
    "1.5 dpf": "../../01-1.5dpf/adatas/anndata_annotated.h5ad",
    "2 dpf": "../../02-2dpf/adatas/anndata_annotated.h5ad",
    "3 dpf": "../../03-3dpf/adatas/anndata_annotated.h5ad",
    "5 dpf": "../../04-5dpf/adatas/anndata_annotated.h5ad",
    "14 dpf": "../../05-14dpf/adatas/anndata_annotated.h5ad",
    "60 dpf": "../../06-60dpf/adatas/anndata_annotated.h5ad",
    "150 dpf": "../../07-150dpf/adatas/anndata_annotated.h5ad"
}

# For option 2: Path to a star solo quant directory
path_quant = ""

# For option 3: Directory containing .mtx, barcodes.tsv and genes.tsv
path_mtx = ""

# For option 4: This is the path to the Seurat (.rds, .robj) file
# A list or dict for multiple files e.g. {"rep_1": "1.rds", "rep_2": "2.rds"}.
path_rds = ""

# Optional: Provide name of Seurat layer to be stored in anndata.
# When reading multiple Seurat files set to string to get the same layer from every object
# or set to list or dict to get specified layers from each object.
# In the second case it must be the same datatype as path_rds.
layer = None

batch_label = None  # Name of the column added to adata.obs only if Option 1 or 4 above is a dict.

-------

In [5]:
if sum(map(lambda x: x != "", [path_h5ad, path_quant, path_mtx, path_rds])) != 1:
    del path_h5ad, path_quant, path_mtx, path_rds
    raise ValueError("Please set only one of the above variables. Adjust the cell above and re-run.")

____

### 3.1 - Option 1: Read from h5ad

In [6]:
if path_h5ad:
    adata = assembler.from_h5ad(path_h5ad, label=batch_label, report="01_assembly.png")

/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


___

### 3.2 -  Option 2: Assemble from preprocessing pipeline 'quant' folder

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [7]:
# change to None if your genes.tsv and bracodes.tsv don't have a header
header = 'infer'

# Set up additional sample the information below.
# Follows the scheme:
# <sample name>:<type>:<value>
# E.g.: 
# sample1:condition:room_air
# sample1:timepoint:early
# sample2:timepoint:late
the_10X_yml = []

__________

In [8]:
if path_quant:
    adata = assembler.from_quant(path_quant, the_10X_yml, header=header, report="01_assembly.png")

____

### 3.3 - Option 3: Create an anndata object from .mtx, barcodes.tsv and genes.tsv

In [9]:
# adjust in case of different naming schemes for any of the input files

mtx = '*matrix.mtx*'  # pattern for the file that contains counts
barcodes = '*barcodes.tsv*'  # pattern for the file that contains barcode information
variables = '*genes.tsv*'  # pattern for the file that contains variable information

In [10]:
if path_mtx:
    adata = assembler.from_mtx(path_mtx, mtx=mtx, barcodes=barcodes, variables=variables, report="01_assembly.png")

___

### 3.4 - Option 4: Convert from Seurat to anndata object

In [11]:
# Converting from Seurat to anndata object
if path_rds:
    adata = assembler.from_R(rds_file=path_rds, label=batch_label, layer=layer, report="01_assembly.png")

__________

## 4 - Prepare anndata
Rename or remove observation (.obs) and variable (.var) columns and layers as needed.

The AnnData object contains all information regarding the dataset at hand. It is commonly stored as a `.h5ad` file. As it is central to this single cell analysis here is a broad overview of its key attributes:

Attributes are accessed with `AnnData.<attribute_name>` e.g. `adata.obs`.

| Attribute | Description |
|-----------|-------------|
|`obs`|A table (pandas DataFrame) containing information to each observation. Observations often refer to individual barcodes or wells and are interpreted as cells.|
|`var`|A table (pandas DataFrame) containing information to each variable. Variables are the measurements taken for each cell and are e.g. genes (RNA) or peaks (ATAC).|
|`X`|A (sparse) matrix containing the actual measurement values. It has the shape `obs x var`. It is the main matrix used for calculations and contains read counts interpreted as gene expression (RNA) or chromatin openness (ATAC). Depending on the analysis progress this may be normalized or corrected.|
|`layers`|A dict-like structure containing additional named matrices. This can be used to store different versions of the matrix e.g. 'raw' and 'normalized'.|
|`raw`|A special attribute containing a raw representation of the dataset. A number of Scanpy functions use this implicitly when available, thus we recommend storing the raw data within the `layers` to avoid confusion.|

For more information visit the [AnnData Read the Docs](https://anndata.readthedocs.io/en/latest/index.html).

In [12]:
import pandas as pd

with pd.option_context('display.max_rows', 5,'display.max_columns', None):
    display(adata.obs)
    display(adata.var)
    display(f"AnnData.raw: {'Yes' if adata.raw else 'No'}")
    display(f"AnnData layers: {list(adata.layers.keys())}")

,paper_cluster,filename,sample,timepoint,batch,Paper cell type,Spatial expression domain,total_counts,log1p_total_counts,total_counts_is_mito,log1p_total_counts_is_mito,pct_counts_is_mito,total_counts_is_ribo,log1p_total_counts_is_ribo,pct_counts_is_ribo,doublet_score,predicted_doublet,predicted_sex,n_genes,log1p_n_genes,S_score,G2M_score,phase,leiden,LISI_score_X_pca,LISI_score_X_umap,phase_density,sample_density,leiden_0.1,leiden_0.2,leiden_0.3,leiden_0.4,leiden_0.5,leiden_0.6,leiden_0.7,leiden_0.8,leiden_0.9,leiden_1.0,leiden_1.1,leiden_1.2,leiden_1.3,leiden_1.4,leiden_1.5,leiden_1.6,leiden_1.7,leiden_1.8,leiden_1.9,clustering,MR_symbol_leiden_1.1,SCSA_symbol_leiden_1.1,MR_score_leiden_1.1,SCSA_score_leiden_1.1,MR_ui_leiden_1.1,SCSA_ui_leiden_1.1,1.3_1,MR_symbol_1.3_1,SCSA_symbol_1.3_1,MR_score_1.3_1,SCSA_score_1.3_1,MR_ui_1.3_1,SCSA_ui_1.3_1,MR_symbol_leiden_0.6,SCSA_symbol_leiden_0.6,MR_score_leiden_0.6,SCSA_score_leiden_0.6,MR_ui_leiden_0.6,SCSA_ui_leiden_0.6,MR_symbol_leiden_1.0,SCSA_symbol_leiden_1.0,MR_score_leiden_1.0,SCSA_score_leiden_1.0,MR_ui_leiden_1.0,SCSA_ui_leiden_1.0,MR_symbol_leiden_0.8,SCSA_symbol_leiden_0.8,MR_score_leiden_0.8,SCSA_score_leiden_0.8,MR_ui_leiden_0.8,SCSA_ui_leiden_0.8,1.1_1,MR_symbol_1.1_1,SCSA_symbol_1.1_1,MR_score_1.1_1,SCSA_score_1.1_1,MR_ui_1.1_1,SCSA_ui_1.1_1,1.8_1,1.8_2,1.8_3,1.8_4,1.8_5,MR_symbol_1.8_5,SCSA_symbol_1.8_5,MR_score_1.8_5,SCSA_score_1.8_5,MR_ui_1.8_5,SCSA_ui_1.8_5
AAACCTGGTTCACCTC-1,2,GSM5402428_scRNAseq_Sox10_Cre_bact_BtR_1.5dpf_...,GSM5402428,1.5dpf,rep_1,Mesenchyme,Dorsal,6103.0,8.716700,72.0,4.290460,1.179748,2230.0,7.710205,36.539406,0.056507,False,Male,1755,7.470794,0.268915,0.248664,S,0,1.000000,1.000000,0.368713,0.574144,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,Periosteum,Mesenchyme,Periosteum,Mesenchyme,Periosteum,Mesenchyme,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACCTGTCCCAAGAT-1,4,GSM5402428_scRNAseq_Sox10_Cre_bact_BtR_1.5dpf_...,GSM5402428,1.5dpf,rep_1,Neuronal,NaN,2433.0,7.797291,20.0,3.044523,0.822030,821.0,6.711740,33.744347,0.017997,False,Male,779,6.659294,-0.356754,-0.144525,G1,2,1.000000,1.000000,0.449944,0.094102,2,3,2,3,3,3,3,3,2,3,2,2,2,2,4,3,3,2,3,2,Hyaline cartilage,Hyaline cartilage,Hyaline cartilage,Hyaline cartilage,Hyaline cartilage,Hyaline cartilage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCATCAACACCA-3,9,GSM5402443_scRNAseq_Sox10_Cre_bact_BtR_150dpf_...,GSM5402443,150dpf,rep_3,Teeth,NaN,1300.0,7.170888,85.0,4.454347,6.538462,561.0,6.331502,43.153847,0.059896,False,Male,360,5.888878,-0.007351,-0.015209,G1,0,2.846069,1.898375,0.976483,0.400197,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,16,16,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,17,8,1,1,Stroma,Stroma,Stroma,Stroma,Stroma,Stroma
TTTGTCATCCGAACGC-3,6,GSM5402443_scRNAseq_Sox10_Cre_bact_BtR_150dpf_...,GSM5402443,150dpf,rep_3,Gill cartilage,NaN,3631.0,8.197538,28.0,3.367296,0.771137,1142.0,7.041412,31.451393,0.074581,False,Male,909,6.813445,-0.061421,0.040508,G2M,12,2.078850,2.291138,0.327639,0.583865,2,2,4,7,8,7,8,9,9,13,10,12,10,10,2,9,11,13,11,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13,14,6,18,9,Gill cartilage,Gill cartilage,Gi

,gene_id,is_mito,is_ribo,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,highly_variable,means,dispersions,dispersions_norm
gene_symbol,,,,,,,,,,,,,
ABR (1 of many),ENSDARG00000059587,False,False,63,0.021192,0.020971,97.913907,64.0,4.174387,False,0.021979,0.229586,-0.204814
ACKR2,ENSDARG00000086314,False,False,1,0.000331,0.000331,99.966887,1.0,0.693147,False,0.000479,0.370179,0.119444
...,...,...,...,...,...,...,...,...,...,...,...,...,...
zzef1,ENSDARG00000094380,False,False,56,0.018543,0.018373,98.145695,56.0,4.043051,False,0.018873,0.223878,-0.217979
zzz3,ENSDARG00000077581,False,False,241,0.083444,0.080145,92.019868,252.0,5.533390,False,0.082994,0.249367,-0.159193


'AnnData.raw: No'

"AnnData layers: ['raw', 'norm']"

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [13]:
# .obs column names that should be deleted
drop_obs = [
    "leiden",
    "LISI_score_X_pca",
    "LISI_score_X_umap",
    "phase_density",
    "sample_density",
    "leiden_0.1",
    "leiden_0.2",
    "leiden_0.3",
    "leiden_0.4",
    "leiden_0.5",
    "leiden_0.6",
    "leiden_0.7",
    "leiden_0.8",
    "leiden_0.9",
    "leiden_1.0",
    "leiden_1.1",
    "leiden_1.2",
    "leiden_1.3",
    "leiden_1.4",
    "leiden_1.5",
    "leiden_1.6",
    "leiden_1.7",
    "leiden_1.8",
    "leiden_1.9",
    "MR_symbol_leiden_1.1",
    "SCSA_symbol_leiden_1.1",
    # "MR_score_leiden_1.1",  # 1.5dpf
    "SCSA_score_leiden_1.1",
    "MR_ui_leiden_1.1",
    "SCSA_ui_leiden_1.1",
    "1.3_1",
    # "MR_symbol_1.3_1",  # 2dpf
    "SCSA_symbol_1.3_1",
    "MR_score_1.3_1",
    "SCSA_score_1.3_1",
    "MR_ui_1.3_1",
    "SCSA_ui_1.3_1",
    "MR_symbol_leiden_0.6",
    "SCSA_symbol_leiden_0.6",
    # "MR_score_leiden_0.6",  # 3dpf
    "SCSA_score_leiden_0.6",
    "MR_ui_leiden_0.6",
    "SCSA_ui_leiden_0.6",
    "MR_symbol_leiden_1.0",
    "SCSA_symbol_leiden_1.0",
    # "MR_score_leiden_1.0",  # 5dpf
    "SCSA_score_leiden_1.0",
    "MR_ui_leiden_1.0",
    "SCSA_ui_leiden_1.0",
    "MR_symbol_leiden_0.8",
    "SCSA_symbol_leiden_0.8",
    # "MR_score_leiden_0.8",  # 14dpf
    "SCSA_score_leiden_0.8",
    "MR_ui_leiden_0.8",
    "SCSA_ui_leiden_0.8",
    "1.1_1",
    "MR_symbol_1.1_1",
    "SCSA_symbol_1.1_1",
    # "MR_score_1.1_1",  # 60dpf
    "SCSA_score_1.1_1",
    "MR_ui_1.1_1",
    "SCSA_ui_1.1_1",
    "1.8_1",
    "1.8_2",
    "1.8_3",
    "1.8_4",
    "1.8_5",
    "MR_symbol_1.8_5",
    "SCSA_symbol_1.8_5",
    # "MR_score_1.8_5",  # 150dpf
    "SCSA_score_1.8_5",
    "MR_ui_1.8_5",
    "SCSA_ui_1.8_5"
]

# .obs column names that should be changed. E.g. "old_name": "New Name"
rename_obs = {
    "clustering": "Combined timepoint clustering",
    "MR_score_leiden_1.1": "1.5 dpf pred. cell type", # 1.5 dpf
    "MR_symbol_1.3_1": "2 dpf pred. cell type", # 2 dpf
    "MR_score_leiden_0.6": "3 dpf pred. cell type", # 3 dpf
    "MR_score_leiden_1.0": "5 dpf pred. cell type", # 5 dpf
    "MR_score_leiden_0.8": "14 dpf pred. cell type", # 14 dpf
    "MR_score_1.1_1": "60 dpf pred. cell type", # 60 dpf
    "MR_score_1.8_5": "150 dpf pred. cell type", # 150 dpf

}

# .var column names that should be deleted
drop_var = []

# .var column names that should be changed. E.g. "old_name": "New Name"
rename_var = {}

# Whether to remove the AnnData.raw. Either a boolean or a string to save raw in `adata.layers[<name>]`
allow_raw = False

# adjust layer names. E.g. "old_name": "New Name"
rename_layer = {}

# save adata.X in a layer
keep_X = None

# replace adata.X with another layer
replace_X = "raw"

# layer names that should be kept. 'all' to keep everything or a list of names
keep = []

________

In [14]:
# change obs
obs = adata.obs.copy()

obs.drop(columns=drop_obs, inplace=True)
obs.rename(columns=rename_obs, errors='raise', inplace=True)

# change var
var = adata.var.copy()

var.drop(columns=drop_var, inplace=True)
var.rename(columns=rename_var, errors='raise', inplace=True)

# apply changes to adata
adata.obs = obs
adata.var = var

# edit adata layers
utils.adata.tidy_layers(adata, allow_raw=allow_raw, rename=rename_layer, keep_X=keep_X, replace_X=replace_X, keep=keep)

In [15]:
# add timepoint suffix to clusters, otherwise e.g. 1 (14dpf) and 1 (150dpf) are merged
adata.obs["Combined timepoint clustering"] = adata.obs["Combined timepoint clustering"].astype(str) + "_" + adata.obs["timepoint"].astype(str)

In [16]:
# combine cell types per timepoint
adata.obs["Combined pred. cell type"] = adata.obs[list(rename_obs.values())[1:]].stack().groupby(level=0).first()

In [17]:
# make cell labels unique
adata.obs.index = [f"{i}-{suffix}" for i, suffix in zip(adata.obs.index, adata.obs["timepoint"])]

_________

## 5 - Saving the loaded anndata object

In [18]:
# Overview of loaded adata
display(adata)

AnnData object with n_obs × n_vars = 51409 × 25904
    obs: 'paper_cluster', 'filename', 'sample', 'timepoint', 'batch', 'Paper cell type', 'Spatial expression domain', 'total_counts', 'log1p_total_counts', 'total_counts_is_mito', 'log1p_total_counts_is_mito', 'pct_counts_is_mito', 'total_counts_is_ribo', 'log1p_total_counts_is_ribo', 'pct_counts_is_ribo', 'doublet_score', 'predicted_doublet', 'predicted_sex', 'n_genes', 'log1p_n_genes', 'S_score', 'G2M_score', 'phase', 'Combined timepoint clustering', '1.5 dpf pred. cell type', '2 dpf pred. cell type', '3 dpf pred. cell type', '5 dpf pred. cell type', '14 dpf pred. cell type', '60 dpf pred. cell type', '150 dpf pred. cell type', 'Combined pred. cell type'
    var: 'gene_id', 'is_mito', 'is_ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'sctoolbox'
    obsm: 'X_pca', 'X_umap'

In [19]:
# Saving the data
adata_output = "anndata_1.h5ad"
utils.adata.save_h5ad(adata, adata_output, report=["01_dataset_overview.md", "01_obs.png", "01_var.png"])

[INFO] The adata object was saved to: ../adatas/anndata_1.h5ad


In [20]:
sctoolbox.settings.close_logfile()